In [ ]:
import pandas as pd

# 1. THE BUSINESS PROBLEM

In [ ]:
stores_df = pd.read_csv('dim_stores.csv')
skus_df = pd.read_csv('dim_skus.csv')
suppliers_df = pd.read_csv(
    'dim_suppliers.csv',
    na_values=['N/A', 'missing', '--', 'NA', 'null'],
    keep_default_na=True
)
events_df = pd.read_csv('dim_events.csv')
fact_df = pd.read_csv('fact_inventory_daily.csv')

In [ ]:
print("--- Data Loading Verification ---")
print(f"Stores dimension: {len(stores_df)} rows")
print(f"SKUs dimension: {len(skus_df)} rows")
print(f"Suppliers dimension: {len(suppliers_df)} rows")
print(f"Events dimension: {len(events_df)} rows")
print(f"Daily Inventory Fact table: {len(fact_df)} rows")
print("-" * 30)

--- Data Loading Verification ---
Stores dimension: 12 rows
SKUs dimension: 60 rows
Suppliers dimension: 15 rows
Events dimension: 30 rows
Daily Inventory Fact table: 21600 rows
------------------------------


In [ ]:
print("\n--- Target Variable Distribution (stockout_risk) ---")

# Calculate row counts and percentage shares
target_counts = fact_df['stockout_risk'].value_counts()
target_percentages = fact_df['stockout_risk'].value_counts(normalize=True) * 100

# Combine into a clean DataFrame for display
distribution_df = pd.DataFrame({
    'Rows': target_counts,
    'Share of Data (%)': target_percentages.round(2)
})

print(distribution_df)
print("\nThis confirms the 3-class imbalance (majority 'Safe', rare high-cost 'Imminent').")


--- Target Variable Distribution (stockout_risk) ---
                Rows  Share of Data (%)
stockout_risk                          
Safe           14131              65.42
At-Risk         5186              24.01
Imminent        2283              10.57

This confirms the 3-class imbalance (majority 'Safe', rare high-cost 'Imminent').


# 2. Schema - 5 Tables

In [ ]:
# Fix the planted data-quality issue: Inconsistent city casing in 'city_display'
stores_df['city_display'] = stores_df['city_display'].str.title()

# Convert date columns to datetime objects for accurate merging and time-series operations
fact_df['date'] = pd.to_datetime(fact_df['date'])
events_df['date'] = pd.to_datetime(events_df['date'])

In [ ]:
# Merge the 5 tables into a single analytical dataframe
# We start with the fact table and left-join dimensions to preserve the exact daily grain
merged_df = (
    fact_df.merge(stores_df, on='store_id', how='left')
           # Include supplier_id here to prevent pandas from creating _x and _y columns
           .merge(skus_df, on=['sku_id', 'supplier_id'], how='left')
           .merge(suppliers_df, on='supplier_id', how='left')
           .merge(events_df, on='date', how='left')
)

In [ ]:
# The spec mandates exactly 21,600 rows after the join.
print("\n--- Schema Join Sanity Check ---")
print(f"Fact table rows before merge: {len(fact_df)}")
print(f"Master table rows after merge:  {len(merged_df)}")

if len(merged_df) == 21600:
    print("Join successful! Row count matches the 21,600 sanity check.")
else:
    print("Warning: Row count mismatch. A join may have caused duplication.")


--- Schema Join Sanity Check ---
Fact table rows before merge: 21600
Master table rows after merge:  21600
Join successful! Row count matches the 21,600 sanity check.


In [ ]:
# Display the first few rows to verify the merged structure
merged_df.head()

,date,store_id,sku_id,supplier_id,opening_stock,units_demanded,units_sold,closing_stock,reorder_point,reorder_placed,...,popularity_tier,supplier_name,categories_supplied,reliability_score,base_lead_time_days,lead_time_variance_days,event_name,event_type,demand_multiplier_festive,demand_multiplier_other
0,2026-10-01,ST01,SKU001,SUP11,159.9,15,15.0,144.9,36.4,N,...,Low,Coastal Foods Pvt Ltd,"Snacks, Fruits & Vegetables",0.83,1,2.5,Regular Day,none,1.0,1.0
1,2026-10-02,ST01,SKU001,SUP11,144.9,10,10.0,134.9,36.4,N,...,Low,Coastal Foods Pvt Ltd,"Snacks, Fruits & Vegetables",0.83,1,2.5,Regular Day,none,1.0,1.0
2,2026-10-03,ST01,SKU001,SUP11,134.9,15,15.0,119.9,36.4,N,...,Low,Coastal Foods Pvt Ltd,"Snacks, Fruits & Vegetables",0.83,1,2.5,Regular Day,none,1.0,1.0
3,2026-10-04,ST01,SKU001,SUP11,119.9,10,10.0,109.9,36.4,N,...,Low,Coastal Foods Pvt Ltd,"Snacks, Fruits & Vegetables",0.83,1,2.5,Regular Day,none,1.0,1.0
4,2026-10-05,ST01,SKU001,SUP11,109.9,9,9.0,100.9,36.4,N,...,Low,Coastal Foods Pvt Ltd,"Snacks, Fruits & Vegetables",0.83,1,2.5,Regular Day,none,1.0,1.0


# 3. Locked Sanity Numbers

In [ ]:
# --- Row Counts ---
print("Row counts:")
print(f"  dim_stores.csv:            {len(stores_df)} rows")
print(f"  dim_skus.csv:               {len(skus_df)} rows")
print(f"  dim_suppliers.csv:          {len(suppliers_df)} rows")
print(f"  dim_events.csv:             {len(events_df)} rows")
print(f"  fact_inventory_daily.csv:   {len(fact_df):,} rows  (12 x 60 x 30)")

# --- Target Distribution ---
print("\nTarget distribution (fact_inventory_daily.csv):")
counts = merged_df['stockout_risk'].value_counts()
pcts = merged_df['stockout_risk'].value_counts(normalize=True) * 100
print(f"  Safe:      {pcts['Safe']:.2f}%  ({counts['Safe']:,} rows)")
print(f"  At-Risk:   {pcts['At-Risk']:.2f}%  ({counts['At-Risk']:,} rows)")
print(f"  Imminent:  {pcts['Imminent']:.2f}%  ({counts['Imminent']:,} rows)")

# --- Calendar Information ---
min_date = merged_df['date'].dt.date.min()
max_date = merged_df['date'].dt.date.max()
unique_dates = merged_df['date'].nunique()
print(f"\nDate range: {min_date} to {max_date} ({unique_dates} unique dates)")

diwali_events = events_df[events_df['event_name'] == 'Diwali Week']
diwali_peak = diwali_events['demand_multiplier_festive'].max()
print(f"Diwali Week: Oct 22-26, 2026 (demand multiplier peaks at {diwali_peak}x on Oct 25)")

flash_sale = events_df[events_df['event_name'] == 'Weekend Flash Sale']
flash_demand = flash_sale['demand_multiplier_other'].max()
print(f"Weekend Flash Sale: Oct 11, 2026 ({flash_demand}x demand, all categories)")

Row counts:
  dim_stores.csv:            12 rows
  dim_skus.csv:               60 rows
  dim_suppliers.csv:          15 rows
  dim_events.csv:             30 rows
  fact_inventory_daily.csv:   21,600 rows  (12 x 60 x 30)

Target distribution (fact_inventory_daily.csv):
  Safe:      65.42%  (14,131 rows)
  At-Risk:   24.01%  (5,186 rows)
  Imminent:  10.57%  (2,283 rows)

Date range: 2026-10-01 to 2026-10-30 (30 unique dates)
Diwali Week: Oct 22-26, 2026 (demand multiplier peaks at 2.6x on Oct 25)
Weekend Flash Sale: Oct 11, 2026 (1.3x demand, all categories)


In [ ]:
# --- Money-Moment Signals ---
print("\nMoney-moment signals (verify these before building your report):\n")

# Helper function to calculate Imminent rate for a given subset
def calc_imminent_rate(condition):
    subset = merged_df[condition]
    return (subset['stockout_risk'] == 'Imminent').mean() * 100

# 1. Festival week effect
non_fest_rate = calc_imminent_rate(merged_df['is_festival_week'] == 'N')
fest_rate = calc_imminent_rate(merged_df['is_festival_week'] == 'Y')
print("Festival week effect:")
print(f"  Imminent rate, non-festival days:  {non_fest_rate:.2f}%")
print(f"  Imminent rate, festival week:      {fest_rate:.2f}%   ({fest_rate/non_fest_rate:.2f}x spike)\n")

# 2. Supplier reliability effect
low_rel = calc_imminent_rate(merged_df['reliability_score'] < 0.75)
mid_rel = calc_imminent_rate((merged_df['reliability_score'] >= 0.75) & (merged_df['reliability_score'] < 0.85))
high_rel = calc_imminent_rate(merged_df['reliability_score'] >= 0.85)

print("Supplier reliability effect:")
print(f"  Imminent rate, low reliability  (<0.75):     {low_rel:.2f}%")
print(f"  Imminent rate, mid reliability (0.75-0.85):   {mid_rel:.2f}%")
print(f"  Imminent rate, high reliability (>=0.85):     {high_rel:.2f}%\n")

# 3. Perishable effect
rate_non_perishable = calc_imminent_rate(merged_df['is_perishable'] == 'N')
rate_perishable = calc_imminent_rate(merged_df['is_perishable'] == 'Y')

print("Perishable effect (smaller but real):")
print(f"  Imminent rate, non-perishable SKUs:  {rate_non_perishable:.1f}%")
print(f"  Imminent rate, perishable SKUs:     {rate_perishable:.1f}%\n")

# --- Missing Data Verification ---
missing_lead_time = merged_df['lead_time_days_actual'].isna().sum()
missing_pct = (missing_lead_time / len(merged_df)) * 100
print(f"Missing lead_time_days_actual: {missing_lead_time:,} of {len(merged_df):,} rows ({missing_pct:.1f}%)")
print("  — by construction, only populated on days a reorder was placed")


Money-moment signals (verify these before building your report):

Festival week effect:
  Imminent rate, non-festival days:  9.51%
  Imminent rate, festival week:      23.31%   (2.45x spike)

Supplier reliability effect:
  Imminent rate, low reliability  (<0.75):     15.83%
  Imminent rate, mid reliability (0.75-0.85):   3.26%
  Imminent rate, high reliability (>=0.85):     3.82%

Perishable effect (smaller but real):
  Imminent rate, non-perishable SKUs:  9.3%
  Imminent rate, perishable SKUs:     12.8%

Missing lead_time_days_actual: 19,899 of 21,600 rows (92.1%)
  — by construction, only populated on days a reorder was placed


# 4. Planted Data-Quality Issues

In [ ]:
# Issue 1: Inconsistent city casing
# We already applied .str.title() during the schema load, but let's verify it worked.
# If we hadn't fixed it, grouping by city_display would split 'BENGALURU' and 'Bengaluru'.
print("--- Verifying City Casing Fix ---")
print("Unique cities in dataset:", merged_df['city_display'].unique())

--- Verifying City Casing Fix ---
Unique cities in dataset: ['Mumbai' 'Bengaluru' 'Delhi' 'Pune' 'Hyderabad' 'Chennai']


In [ ]:
# ISSUE 2: The 'N/A' reliability score trap
# We successfully parsed the literal text 'N/A' into true NaNs using the na_values
# parameter during pd.read_csv() earlier.
# Now, we resolve these missing values by imputing them with the category median.

print("\n--- Imputing Missing Reliability Scores ---")
missing_before = merged_df['reliability_score'].isna().sum()
print(f"Missing reliability scores before imputation: {missing_before}")

# Create the clean feature by grouping by 'category' and filling NaNs with the median
merged_df['supplier_reliability_clean'] = (
    merged_df.groupby('category')['reliability_score']
             .transform(lambda x: x.fillna(x.median()))
)

missing_after = merged_df['supplier_reliability_clean'].isna().sum()
print(f"Missing reliability scores after imputation:  {missing_after}")

# Let's peek at a few rows where the original score was missing to see the imputed values
imputed_sample = merged_df[merged_df['reliability_score'].isna()][
    ['category', 'supplier_id', 'reliability_score', 'supplier_reliability_clean']
].drop_duplicates()

print("\nSample of imputed values based on category medians:")
print(imputed_sample.head())


--- Imputing Missing Reliability Scores ---
Missing reliability scores before imputation: 5040
Missing reliability scores after imputation:  0

Sample of imputed values based on category medians:
            category supplier_id  reliability_score  \
10440      Beverages       SUP15                NaN   
11880  Personal Care       SUP06                NaN   
14400      Home Care       SUP06                NaN   
16920        Staples       SUP14                NaN   

       supplier_reliability_clean  
10440                        0.70  
11880                        0.69  
14400                        0.67  
16920                        0.91  


# 5. Feature Engineering

In [ ]:
# 1. Basic Derived Features
# How far past or before the trigger point are we?
merged_df['reorder_gap'] = merged_df['reorder_point'] - merged_df['closing_stock']

# Normalized cover: ratio of days of cover to expected lead time
merged_df['days_of_cover_ratio'] = merged_df['days_of_cover'] / merged_df['lead_time_days_expected']

In [ ]:
# 2. Sequential/Rolling Features
# Flag if a reorder was placed within the last N days (Let's use N=3)
# First, sort values to ensure chronological order for the shift/rolling operations
merged_df = merged_df.sort_values(['store_id', 'sku_id', 'date']).reset_index(drop=True)

# Convert 'reorder_placed' (Y/N) to binary (1/0)
merged_df['reorder_placed_bin'] = (merged_df['reorder_placed'] == 'Y').astype(int)

# Group by store and SKU, then look at the previous 3 days to see if an order was placed
merged_df['is_recent_reorder'] = (
    merged_df.groupby(['store_id', 'sku_id'])['reorder_placed_bin']
             .transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).max())
).fillna(0).astype(int)

In [ ]:
# 3. Temporal / Calendar Features
# Extract day of the month
merged_df['day_of_month'] = merged_df['date'].dt.day

# Days since festival start (Diwali week starts Oct 22, 2026)
festival_start = pd.to_datetime('2026-10-22')
# This will be negative for days before the festival (acting as a countdown) and positive during/after
merged_df['days_since_festival_start'] = (merged_df['date'] - festival_start).dt.days

In [ ]:
# 4. Categorical Encoding
# One-hot encoding the 8 product categories
merged_df = pd.get_dummies(merged_df, columns=['category'], prefix='cat', drop_first=False)

# Optional: Map the target variable to numeric classes for ML algorithms
target_mapping = {'Safe': 0, 'At-Risk': 1, 'Imminent': 2}
merged_df['target_class'] = merged_df['stockout_risk'].map(target_mapping)

In [ ]:
# 5. The Train / Test Split (Time-based to prevent leakage)
# The spec strictly warns against random row-level splits for daily panel data.
# We will train on Oct 1-23 and test on the generalization challenge of Oct 24-30.

train_mask = merged_df['date'] <= '2026-10-23'
test_mask = merged_df['date'] >= '2026-10-24'

train_df = merged_df[train_mask].copy()
test_df = merged_df[test_mask].copy()

print("--- Train / Test Split Completed ---")
print(f"Training set (Oct 1-23):  {len(train_df):,} rows")
print(f"Testing set (Oct 24-30):   {len(test_df):,} rows")

# Prepare X and y for modeling
features_to_drop = [
    'date', 'store_id', 'sku_id', 'supplier_id', 'supplier_name',
    'city', 'city_display', 'event_name', 'stockout_risk', 'target_class',
    'reorder_placed', 'is_perishable', 'festive_relevant', 'popularity_tier',
    'store_size', 'event_type', 'is_festival_week'
]

X_train = train_df.drop(columns=features_to_drop, errors='ignore')
y_train = train_df['target_class']

X_test = test_df.drop(columns=features_to_drop, errors='ignore')
y_test = test_df['target_class']

print(f"\nFeature Matrix Shape: {X_train.shape[1]} columns ready for modeling.")

--- Train / Test Split Completed ---
Training set (Oct 1-23):  16,560 rows
Testing set (Oct 24-30):   5,040 rows

Feature Matrix Shape: 37 columns ready for modeling.


In [ ]:
import os
import pandas as pd

# Define the expected manifest from the project spec
expected_manifest = {
    'dim_stores.csv': {'rows': 12, 'purpose': 'Store dimension'},
    'dim_skus.csv': {'rows': 60, 'purpose': 'Product dimension'},
    'dim_suppliers.csv': {'rows': 15, 'purpose': 'Supplier dimension'},
    'dim_events.csv': {'rows': 30, 'purpose': 'Festival/promo calendar'},
    'fact_inventory_daily.csv': {'rows': 21600, 'purpose': 'Modeling table - join the above 4 tables to this one'}
}

print("--- QuickCart Stockout Risk: File Manifest Check ---\n")
print(f"{'Filename':<26} | {'Status':<10} | {'Expected':<8} | {'Actual':<8} | {'Purpose'}")
print("-" * 90)

# Iterate through the expected files and verify them against the local directory
for filename, info in expected_manifest.items():
    expected_rows = info['rows']
    purpose = info['purpose']

    if os.path.exists(filename):
        # Read just the length of the file to save memory
        actual_rows = len(pd.read_csv(filename))
        status = " PASS" if actual_rows == expected_rows else " FAIL"
    else:
        actual_rows = "N/A"
        status = " MISSING"

    print(f"{filename:<26} | {status:<10} | {expected_rows:<8} | {actual_rows:<8} | {purpose}")

print("\nContinuity Note Authenticated: QuickCart is confirmed as the same brand used in Week 8 Session 1 (K-Means customer clustering), but utilizing a completely separate, unrelated dataset with no shared keys.")

--- QuickCart Stockout Risk: File Manifest Check ---

Filename                   | Status     | Expected | Actual   | Purpose
------------------------------------------------------------------------------------------
dim_stores.csv             |  PASS      | 12       | 12       | Store dimension
dim_skus.csv               |  PASS      | 60       | 60       | Product dimension
dim_suppliers.csv          |  PASS      | 15       | 15       | Supplier dimension
dim_events.csv             |  PASS      | 30       | 30       | Festival/promo calendar
fact_inventory_daily.csv   |  PASS      | 21600    | 21600    | Modeling table - join the above 4 tables to this one

Continuity Note Authenticated: QuickCart is confirmed as the same brand used in Week 8 Session 1 (K-Means customer clustering), but utilizing a completely separate, unrelated dataset with no shared keys.
